In [38]:
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_cohere import CohereEmbeddings, ChatCohere
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from dotenv import load_dotenv
load_dotenv()

True

In [39]:
# Sample documents for testing
documents = [
    # --- Climate (3) ---
    Document(
        page_content="Global temperatures have risen significantly over the past century due to increased greenhouse gas emissions. Climate change is leading to more frequent extreme weather events, including heatwaves, floods, and hurricanes.",
        metadata={"topic": "climate"}
    ),
    Document(
        page_content="Melting polar ice caps and glaciers are contributing to rising sea levels. Coastal communities around the world face increasing risks of flooding and displacement.",
        metadata={"topic": "climate"}
    ),
    Document(
        page_content="Transitioning to renewable energy sources such as solar and wind power is essential to reducing carbon emissions and mitigating the long-term impacts of climate change.",
        metadata={"topic": "climate"}
    ),

    # --- Health (3) ---
    Document(
        page_content="Regular physical activity reduces the risk of chronic diseases such as heart disease, diabetes, and obesity. Experts recommend at least 150 minutes of moderate exercise per week.",
        metadata={"topic": "health"}
    ),
    Document(
        page_content="Mental health awareness has increased globally, highlighting the importance of early intervention, therapy, and reducing the stigma associated with psychological disorders.",
        metadata={"topic": "health"}
    ),
    Document(
        page_content="Advancements in medical technology, including telemedicine and AI-driven diagnostics, are improving access to healthcare and enabling earlier detection of diseases.",
        metadata={"topic": "health"}
    ),

    # --- Current Geopolitics (3) ---
    Document(
        page_content="Rising tensions in Eastern Europe continue to influence global security policies, with NATO members increasing defense spending and diplomatic efforts ongoing to prevent further escalation.",
        metadata={"topic": "geopolitics"}
    ),
    Document(
        page_content="Trade relations between major economies remain complex, with supply chain diversification and economic sanctions shaping international commerce.",
        metadata={"topic": "geopolitics"}
    ),
    Document(
        page_content="Shifts in energy policy are redefining alliances, as countries seek to secure critical minerals and alternative energy sources amid global competition.",
        metadata={"topic": "geopolitics"}
    ),

    # --- Sport (1) ---
    Document(
        page_content="The global popularity of football continues to grow, with international tournaments drawing millions of viewers and significantly impacting sports economics worldwide.",
        metadata={"topic": "sport"}
    ),
]

In [40]:
#Initialize the Cohere embeddings and FAISS vector store

embedding_model = CohereEmbeddings(model="multilingual-22-12")

vector_store = FAISS.from_documents(
    documents=documents,
    embedding=embedding_model
)

In [41]:
#Initialize the multi-query retriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(search_kwargs={"k": 2}),
    llm=ChatCohere(model="command-a-03-2025")
)

In [42]:
#Query

query = "How to be healthy?"

In [ ]:
# Getting relevant documents using the multi-query retriever

relevant_docs = multi_query_retriever.invoke(query)

# The MultiQueryRetriever generates multiple reformulated queries from your original query, 
# then retrieves k documents for each reformulated query. Results are deduplicated but you still get more documents total.

# Example: If it generates 3 reformulated queries and k=2, you could get up to 6 documents (3 queries × 2 docs each, minus any duplicates).

# To limit final output to exactly 2 documents, add this after retrieval:

# Limit to top 2 documents
# relevant_docs = relevant_docs[:2]

In [44]:
for doc in relevant_docs:
    print(f"Content: {doc.page_content}\nMetadata: {doc.metadata}\n")

Content: Advancements in medical technology, including telemedicine and AI-driven diagnostics, are improving access to healthcare and enabling earlier detection of diseases.
Metadata: {'topic': 'health'}

Content: Trade relations between major economies remain complex, with supply chain diversification and economic sanctions shaping international commerce.
Metadata: {'topic': 'geopolitics'}

Content: Mental health awareness has increased globally, highlighting the importance of early intervention, therapy, and reducing the stigma associated with psychological disorders.
Metadata: {'topic': 'health'}

